# 07 — Análisis de Robustez y Quality Gating

Dos preguntas de producción:

1. **Robustez**: ¿cuánto se degrada el modelo cuando la imagen de entrada tiene problemas de captura (desenfoque, compresión, ruido, baja resolución, subexposición)?
2. **Quality gating**: ¿podemos detectar automáticamente esas imágenes malas y rechazarlas *antes* de la inferencia, para no reportar predicciones poco fiables?

Usamos `src.preprocessing.degradations` para simular cada problema a severidad creciente y `src.preprocessing.quality` para el gate.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from src.preprocessing.degradations import DegradationType, apply_degradation
from src.preprocessing.quality import ImageQualityAssessor, QualityConfig

## 1. Imagen de muestra

Cargamos una muestra real de `data/samples/` (o generamos una sintética con textura si no hay).

In [ ]:
sample_dir = ROOT / 'data' / 'samples'
samples = sorted(sample_dir.glob('*.jpg')) + sorted(sample_dir.glob('*.png'))

if samples:
    img = cv2.cvtColor(cv2.imread(str(samples[0])), cv2.COLOR_BGR2RGB)
    print(f'Cargada: {samples[0].name}  shape={img.shape}')
else:
    # Synthetic textured document: text-like horizontal bars on light background
    img = np.full((400, 600, 3), 235, dtype=np.uint8)
    rng = np.random.default_rng(0)
    for y in range(40, 360, 28):
        w = rng.integers(200, 520)
        img[y:y+10, 40:40+w] = rng.integers(20, 90)
    print(f'Generada sintética  shape={img.shape}')

## 2. Galería de degradaciones

Cada degradación a severidad 0.6 para inspección visual.

In [ ]:
degradations = list(DegradationType)
fig, axes = plt.subplots(1, len(degradations) + 1, figsize=(3 * (len(degradations) + 1), 3.5))

axes[0].imshow(img)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for ax, deg in zip(axes[1:], degradations):
    degraded = apply_degradation(img, deg, severity=0.6)
    ax.imshow(degraded)
    ax.set_title(deg.value, fontsize=9)
    ax.axis('off')

plt.tight_layout()
out_dir = ROOT / 'reports' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / '07_degradation_gallery.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Métricas de calidad vs severidad

Para cada degradación, barremos severidad 0→1 y registramos las métricas del `ImageQualityAssessor`.  
Esto valida que el gate reacciona en la dirección correcta a cada tipo de problema.

In [ ]:
assessor = ImageQualityAssessor()
severities = np.linspace(0, 1, 11)

records = []
for deg in degradations:
    for s in severities:
        degraded = apply_degradation(img, deg, severity=float(s))
        report = assessor.assess(degraded)
        records.append({
            'degradation': deg.value,
            'severity': s,
            'sharpness': report.sharpness,
            'brightness': report.brightness,
            'contrast': report.contrast,
            'clipping': report.clipping,
            'passed': report.passed,
        })

df = pd.DataFrame(records)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for deg in degradations:
    sub = df[df.degradation == deg.value]
    axes[0].plot(sub.severity, sub.sharpness, marker='o', label=deg.value, ms=4)
    axes[1].plot(sub.severity, sub.brightness, marker='o', label=deg.value, ms=4)
    axes[2].plot(sub.severity, sub.contrast, marker='o', label=deg.value, ms=4)

axes[0].axhline(QualityConfig().min_sharpness, ls='--', color='red', lw=1, label='umbral')
axes[0].set_title('Nitidez (var. Laplaciano)')
axes[0].set_yscale('log')
axes[1].set_title('Brillo medio')
axes[2].set_title('Contraste (std)')

for ax in axes:
    ax.set_xlabel('severidad')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(out_dir / '07_quality_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Curva de rechazo del gate

¿A qué severidad empieza el gate a rechazar cada degradación?

In [ ]:
pivot = df.pivot_table(index='severity', columns='degradation', values='passed')

fig, ax = plt.subplots(figsize=(9, 4))
for deg in degradations:
    ax.step(pivot.index, pivot[deg.value].astype(int), where='post', marker='o', label=deg.value, ms=4)

ax.set_xlabel('severidad')
ax.set_ylabel('passed (1=aceptada, 0=rechazada)')
ax.set_title('Decisión del quality gate vs severidad de degradación')
ax.set_yticks([0, 1])
ax.legend(fontsize=8, loc='center right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / '07_gate_decision.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Severidad mínima en la que cada degradación es rechazada
first_reject = (
    df[~df.passed]
    .groupby('degradation')['severity']
    .min()
    .reindex([d.value for d in degradations])
)
summary = first_reject.to_frame('severidad_de_rechazo')
summary['severidad_de_rechazo'] = summary['severidad_de_rechazo'].map(
    lambda x: f'{x:.1f}' if pd.notna(x) else 'nunca (en rango 0–1)'
)
summary

## 5. Conclusiones

- **El gate detecta desenfoque y baja resolución de forma temprana** — la varianza del Laplaciano cae por debajo del umbral con poco blur, y el `downscale` reduce la resolución efectiva.
- **La subexposición (`brightness`) se captura por el umbral de brillo medio.**
- **La compresión JPEG y el ruido gaussiano son más sutiles** para métricas no-referenciadas: degradan la predicción del modelo más que las métricas de calidad clásicas. Para producción conviene complementar con un detector de artefactos de bloque (DCT) si la compresión es un problema frecuente.

**Recomendación de producción**: activar `check_quality=True` en la API. Las imágenes rechazadas (`label='rejected'`) deben re-capturarse en lugar de forzar una predicción no fiable. Los umbrales de `QualityConfig` están calibrados para fotos de documentos y son ajustables según el canal de captura (escáner vs móvil).